This layer will give insights on business KPIs like:

- total claims per payer
- approval rate 
- avg claim cost per specialty
- patient utilization
- paid vs pending claims
- procedure cost distribution

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

silver = "healthcare_adjudication.healthcare_adjudication_silver"
gold   = "healthcare_adjudication.healthcare_adjudication_gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold}")

In [0]:
# Which payer receives most claims?
policies_current = spark.table(f"{silver}.policies").filter(col("is_current") == True)
claims_by_payer = (
    spark.table(f"{silver}.claims").alias("c")
    .join(policies_current.alias("p"), "policy_id")
    .join(spark.table(f"{silver}.payers").alias("py"), "payer_id")
    .groupBy("payer_id","payer_name")
    .agg(
        count("*").alias("total_claims"),
        sum("claim_amount").alias("total_claim_amount")
    )
)
claims_by_payer.repartition(8).write.format("delta").mode("overwrite") \
.saveAsTable(f"{gold}.claims_by_payer")

In [0]:
#What % of claims are approved vs denied?
approval_rate = (
    spark.table(f"{silver}.claims")
    .groupBy("status")
    .agg(count("*").alias("claims"))
    .withColumn(
        "percentage",
        round(col("claims")/sum("claims").over(Window.partitionBy())*100,2)
    )
)
approval_rate.repartition(8).write.format("delta").mode("overwrite") \
.saveAsTable(f"{gold}.claim_approval_rate")

In [0]:
#Which specialties cost most?
avg_cost_specialty = (
    spark.table(f"{silver}.claims").alias("c")
    .join(spark.table(f"{silver}.providers").alias("pr"), "provider_id")
    .groupBy("specialty")
    .agg(
        round(avg("claim_amount"),2).alias("avg_claim_cost"),
        sum("claim_amount").alias("total_cost")
    )
)

avg_cost_specialty.write.format("delta").mode("overwrite") \
.saveAsTable(f"{gold}.avg_cost_by_specialty")


In [0]:
#How much of claim value actually paid?
payment_vs_claim = (
    spark.table(f"{silver}.claims").alias("c")
    .join(spark.table(f"{silver}.payments").alias("p"), "claim_id","left")
    .groupBy("claim_id","claim_amount")
    .agg(sum("amount").alias("total_paid"))
    .withColumn("paid_ratio", col("total_paid")/col("claim_amount"))
)

payment_vs_claim.write.format("delta").mode("overwrite") \
.saveAsTable(f"{gold}.payment_vs_claim")


In [0]:
#Which procedures drive costs?
procedure_cost = (
    spark.table(f"{silver}.claim_lines")
    .groupBy("procedure_code")
    .agg(
        count("*").alias("num_times_used"),
        sum("procedure_charge").alias("total_cost"),
        round(avg("procedure_charge"),2).alias("avg_cost")
    )
)

procedure_cost.write.format("delta").mode("overwrite") \
.saveAsTable(f"{gold}.procedure_cost_distribution")


In [0]:
#Claims over time
monthly_claims = (
    spark.table(f"{silver}.claims")
    .withColumn("month", date_trunc("month","claim_date"))
    .groupBy("month")
    .agg(
        count("*").alias("num_claims"),
        sum("claim_amount").alias("total_claim_amount")
    )
)

monthly_claims.write.format("delta").mode("overwrite") \
.saveAsTable(f"{gold}.monthly_claim_trend")
